Vanessa Wafa Wehbe

1. Download All Data (CSVs & Images)

In [2]:
# Download text metadata
!wget -q https://uni-bonn.sciebo.de/s/r7iYfS2QTDrA4CD/download/labels.csv -O labels.csv
!wget -q https://uni-bonn.sciebo.de/s/tjc7eF5CsynKwrG/download/train_labels.csv -O train_labels.csv
!wget -q https://uni-bonn.sciebo.de/s/KFyAPTgjqMPWiBA/download/test_labels.csv -O test_labels.csv
!wget -q https://uni-bonn.sciebo.de/s/f7yaDakFNS5C2n7/download/metadata_clean.csv -O metadata_clean.csv

# Download and extract the image ZIP file
!wget -q https://uni-bonn.sciebo.de/s/KrMiTk2X7sgBCwK/download/MIMIC-CXR-png.zip -O MIMIC-CXR-png.zip
!unzip -q -o MIMIC-CXR-png.zip -d /content/images

2. Setup & Imports

In [3]:
import os
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from transformers import AutoProcessor, AutoModel
from peft import get_peft_model, LoraConfig
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

3. Data Merging

In [4]:
# Merge labels with metadata based on image ID
train_labels_raw = pd.read_csv('train_labels.csv')
test_labels_raw = pd.read_csv('test_labels.csv')
metadata = pd.read_csv('metadata_clean.csv')

train_df_full = train_labels_raw.merge(metadata, on='dicom_id', how='left')
test_df = test_labels_raw.merge(metadata, on='dicom_id', how='left')

train_df_full = train_df_full.dropna(subset=['mask_path']).reset_index(drop=True)
test_df = test_df.dropna(subset=['mask_path']).reset_index(drop=True)

# 80/20 Train-Validation Split
train_df, val_df = train_test_split(train_df_full, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# The root directory containing the nested 'segmentation' folders
image_dir = "/content/images/MIMIC-CXR-png"

4. MIMIC-CXR Dataset Class

In [5]:
tabular_feature_size = 1

class MimicCxrMultimodalDataset(Dataset):
    def __init__(self, df, processor=None, is_medclip=False):
        self.df = df
        self.processor = processor
        self.is_medclip = is_medclip
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = str(row['mask_path'])
        img_path = os.path.join(image_dir, relative_path)
        image = Image.open(img_path).convert('RGB')

        if self.is_medclip and self.processor:
            image_tensor = self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        else:
            image_tensor = self.transform(image)

        view = str(row['ViewCodeSequence_CodeMeaning']).lower()
        is_pa = 1.0 if 'postero-anterior' in view else 0.0
        tabular_tensor = torch.tensor([is_pa], dtype=torch.float32)

        lbl = row['pathology']
        if isinstance(lbl, str) and '[' in lbl:
            lbl = eval(lbl.replace(' ', ','))
        if isinstance(lbl, (list, np.ndarray, tuple)):
            lbl = lbl[0]

        label = torch.tensor(float(lbl), dtype=torch.float32)

        return image_tensor, tabular_tensor, label

processor = AutoProcessor.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")

train_loader_clip = DataLoader(MimicCxrMultimodalDataset(train_df, processor=processor, is_medclip=True), batch_size=32, shuffle=True)
val_loader_clip = DataLoader(MimicCxrMultimodalDataset(val_df, processor=processor, is_medclip=True), batch_size=32, shuffle=False)
test_loader_clip = DataLoader(MimicCxrMultimodalDataset(test_df, processor=processor, is_medclip=True), batch_size=32, shuffle=False)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

5. The 5 Models

In [6]:
class MedClipLora(nn.Module):
    def __init__(self, vision_model, num_tabular_features):
        super().__init__()
        for param in vision_model.parameters(): param.requires_grad = False
        self.vision_model = get_peft_model(vision_model, LoraConfig(r=16, lora_alpha=16, target_modules=["q_proj", "v_proj"]))
        self.classifier = nn.Sequential(nn.Linear(512 + num_tabular_features, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, image, tabular):
        img_features = self.vision_model.get_image_features(pixel_values=image)
        if hasattr(img_features, 'pooler_output'): img_features = img_features.pooler_output
        return self.classifier(torch.cat([img_features, tabular], dim=1))

6. Training & Execution

In [7]:
def train_and_evaluate(model, model_name, train_loader, val_loader, epochs=20):
    print(f"\n--- Training {model_name} ---")
    model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()
    best_auc, best_acc, best_f1 = 0, 0, 0
    best_state_dict = None

    for epoch in range(epochs):
        model.train()
        for images, tabular, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            optimizer.zero_grad()
            loss = criterion(model(images.to(device), tabular.to(device)).squeeze(), labels.to(device))
            loss.backward()
            optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for images, tabular, labels in val_loader:
                outputs = model(images.to(device), tabular.to(device))
                all_probs.extend(torch.sigmoid(outputs.squeeze()).cpu().numpy())
                all_labels.extend(labels.numpy())

        all_preds = (np.array(all_probs) > 0.5).astype(int)

        try:
            val_auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            val_auc = 0.0

        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, zero_division=0)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state_dict = model.state_dict()

        best_acc, best_f1 = max(best_acc, val_acc), max(best_f1, val_f1)
        print(f"Epoch {epoch+1} | Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if best_state_dict is not None:
        torch.save(best_state_dict, f"{model_name.replace(' ', '_')}_best_weights.pth")

    training_time = time.time() - start_time
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else 0

    return {
        'Model': model_name,
        'AUC': best_auc, 'Accuracy': best_acc, 'F1': best_f1,
        'Time (s)': round(training_time, 2),
        'GPU Mem (MB)': round(gpu_mem, 2),
        'Params': f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    }

results = []
epochs = 20

fresh_base = AutoModel.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
medclip_model = MedClipLora(fresh_base, tabular_feature_size)

results.append(train_and_evaluate(medclip_model, "MedCLIP LoRA", train_loader_clip, val_loader_clip, epochs))

print(pd.DataFrame(results).set_index('Model'))

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: flaviagiammarino/pubmed-clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Training MedCLIP LoRA ---


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

Epoch 18 | Val AUC: 0.6492 | Val Acc: 0.6190 | Val F1: 0.6000


Epoch 19:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 19 | Val AUC: 0.6724 | Val Acc: 0.5873 | Val F1: 0.5517


Epoch 20:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 20 | Val AUC: 0.6683 | Val Acc: 0.5873 | Val F1: 0.5517
                   AUC  Accuracy        F1  Time (s)  GPU Mem (MB)     Params
Model                                                                        
MedCLIP LoRA  0.761089   0.68254  0.734177    466.35       1507.79  1,048,961


In [8]:
def save_test_samples(test_loader, num_samples=5, output_dir="test_samples"):
    os.makedirs(output_dir, exist_ok=True)
    images, tabular, labels = next(iter(test_loader))

    for i in range(min(num_samples, len(images))):
        img_tensor = images[i].cpu()

        img_min = img_tensor.min()
        img_max = img_tensor.max()
        img_normalized = (img_tensor - img_min) / (img_max - img_min)

        file_name = os.path.join(output_dir, f"test_sample_{i}_label_{int(labels[i].item())}.png")
        torchvision.utils.save_image(img_normalized, file_name)

    print(f"Saved {min(num_samples, len(images))} sample images to directory: '{output_dir}'")

save_test_samples(test_loader_clip, num_samples=5)

Saved 5 sample images to directory: 'test_samples'
